# Part 3 — Ingestion: PDF Loader & Chunker

Questo notebook dimostra il corretto funzionamento di `src/ingestion/pdf_loader.py` (EP-02, TASK-09).

| Funzione | Descrizione |
|----------|-------------|
| `load_pdf(path)` | Estrae il testo da ogni pagina di un PDF con PyMuPDF |
| `chunk_documents(docs)` | Divide i `Document` in `Chunk` semanticamente coerenti |
| `load_and_chunk_pdf(path)` | Convenience: load + chunk in una sola chiamata |

**Strategia di chunking:** `RecursiveCharacterTextSplitter` con separatori `["\n\n", "\n", ". ", " "]`, dimensione in token tiktoken (`cl100k_base`), `chunk_size=512`, `chunk_overlap=64`.

> Il notebook crea un PDF sintetico in memoria da testo di business glossary di esempio (stessa fonte dei fixture del progetto) tramite PyMuPDF, poi lo carica e chunkizza.

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: /home/marcantoniolopez/Documenti/github/thesis


## 1. Creazione di un PDF sintetico con PyMuPDF

Poiché l'ingestion pipeline opera su PDF, creiamo un documento di test programmaticamente dalla stessa fonte usata dai fixture: `tests/fixtures/sample_docs/business_glossary.txt`. Il PDF viene salvato in una directory temporanea.

In [2]:
import tempfile
import fitz  # pymupdf

# Legge il business glossary di esempio
glossary_path = project_root / "tests" / "fixtures" / "sample_docs" / "business_glossary.txt"
glossary_text = glossary_path.read_text(encoding="utf-8")

print(f"Business glossary: {len(glossary_text)} caratteri, {len(glossary_text.splitlines())} righe")
print("\nPrime 5 righe:")
for line in glossary_text.splitlines()[:5]:
    print(f"  {line}")

Business glossary: 2371 caratteri, 30 righe

Prime 5 righe:
  BUSINESS GLOSSARY — E-COMMERCE DOMAIN
  
  Customer
  A Customer is any individual or legal entity that has registered an account on the platform and has made at least one purchase. Customers are identified by a unique Customer ID and may have one or more associated delivery addresses. Synonyms: client, buyer, account holder.
  


In [3]:
import textwrap

# Suddivide il testo in pagine simulate (una per concetto principale)
# In un PDF reale ogni pagina è estratta dall'ufficio di governance
sections = glossary_text.strip().split("\n\n")
sections = [s.strip() for s in sections if s.strip()]

print(f"Sezioni individuate: {len(sections)}")
for i, s in enumerate(sections, 1):
    print(f"  Sezione {i}: {s[:60].replace(chr(10), ' ')}...")

Sezioni individuate: 9
  Sezione 1: BUSINESS GLOSSARY — E-COMMERCE DOMAIN...
  Sezione 2: Customer A Customer is any individual or legal entity that h...
  Sezione 3: Product A Product is a tangible or digital item offered for ...
  Sezione 4: SalesOrder A SalesOrder is a formal transactional document t...
  Sezione 5: OrderLineItem An OrderLineItem is a single line within a Sal...
  Sezione 6: Category A Category is a hierarchical classification groupin...
  Sezione 7: CustomerAddress A CustomerAddress is a postal or geographic ...
  Sezione 8: Inventory Inventory refers to the quantity of a Product curr...
  Sezione 9: KEY RELATIONSHIPS: - A Customer places one or more SalesOrde...


In [4]:
# Crea il PDF sintetico: una sezione per pagina
tmp_dir = Path(tempfile.mkdtemp())
pdf_path = tmp_dir / "business_glossary.pdf"

doc = fitz.open()  # nuovo documento vuoto
for section in sections:
    page = doc.new_page(width=595, height=842)  # A4
    # Inserisce il testo con wrapping automatico
    rect = fitz.Rect(50, 50, 545, 792)
    page.insert_textbox(
        rect,
        section,
        fontsize=11,
        fontname="helv",
        color=(0, 0, 0),
    )

doc.save(str(pdf_path))
doc.close()

file_size_kb = pdf_path.stat().st_size / 1024
print(f"PDF creato: {pdf_path}")
print(f"Dimensione: {file_size_kb:.1f} KB")
print(f"Pagine: {len(sections)}")

PDF creato: /tmp/tmp68wiqx6l/business_glossary.pdf
Dimensione: 5.7 KB
Pagine: 9


## 2. `load_pdf()` — Estrazione del testo

`load_pdf()` apre il PDF con `fitz.open()`, itera su ogni pagina, estrae il testo con `page.get_text("text")`, e crea un `Document` per ogni pagina non vuota. Gestisce PDF mancanti, corrotti o protetti da password con `IngestionError`.

In [5]:
from src.ingestion.pdf_loader import load_pdf, IngestionError
from src.models.schemas import Document

documents = load_pdf(pdf_path)

print(f"Documenti estratti: {len(documents)}")
print()
for i, doc in enumerate(documents[:4], 1):
    text_preview = doc.text[:80].replace("\n", " ")
    print(f"Document {i}:")
    print(f"  metadata: {doc.metadata}")
    print(f"  text (first 80): '{text_preview}'")
    print()

/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/pydantic_settings/sources/providers/secrets.py:67: UserWarning: directory "/run/secrets" does not exist
  warnings.warn(f'directory "{path}" does not exist')
{"ts": "2026-03-12T10:50:11", "logger": "src.ingestion.pdf_loader", "level": "INFO", "message": "Loaded 9 pages from 'business_glossary.pdf'"}


Documenti estratti: 9

Document 1:
  metadata: {'source': 'business_glossary.pdf', 'page': '1'}
  text (first 80): 'BUSINESS GLOSSARY ? E-COMMERCE DOMAIN'

Document 2:
  metadata: {'source': 'business_glossary.pdf', 'page': '2'}
  text (first 80): 'Customer A Customer is any individual or legal entity that has registered an acc'

Document 3:
  metadata: {'source': 'business_glossary.pdf', 'page': '3'}
  text (first 80): 'Product A Product is a tangible or digital item offered for sale on the platform'

Document 4:
  metadata: {'source': 'business_glossary.pdf', 'page': '4'}
  text (first 80): 'SalesOrder A SalesOrder is a formal transactional document that records the agre'



In [6]:
# Verifica gestione errori

# 1. File inesistente
try:
    load_pdf(Path("/tmp/inesistente.pdf"))
except IngestionError as e:
    print(f"✅ File mancante → IngestionError: {e}")

# 2. Tutti i document hanno i metadati attesi
for doc in documents:
    assert "source" in doc.metadata, "metadata deve contenere 'source'"
    assert "page" in doc.metadata, "metadata deve contenere 'page'"
    assert isinstance(doc, Document)

print(f"✅ Tutti i {len(documents)} document hanno metadati 'source' e 'page'")
print(f"   source: '{documents[0].metadata['source']}'")
print(f"   page range: {documents[0].metadata['page']} → {documents[-1].metadata['page']}")

✅ File mancante → IngestionError: PDF file not found: /tmp/inesistente.pdf
✅ Tutti i 9 document hanno metadati 'source' e 'page'
   source: 'business_glossary.pdf'
   page range: 1 → 9


## 3. `chunk_documents()` — Chunking semantico

`chunk_documents()` usa `RecursiveCharacterTextSplitter` con dimensione in token (tiktoken `cl100k_base`). I parametri provengono da `settings.chunk_size` e `settings.chunk_overlap`. Ogni chunk eredita i metadati del documento padre e aggiunge `token_count`.

In [7]:
from src.ingestion.pdf_loader import chunk_documents
from src.config.settings import settings

print(f"Parametri di chunking:")
print(f"  chunk_size:    {settings.chunk_size} token")
print(f"  chunk_overlap: {settings.chunk_overlap} token")
print()

chunks = chunk_documents(documents)

print(f"Chunks prodotti: {len(chunks)} (da {len(documents)} documenti)")
if chunks:
    token_counts = [int(c.metadata.get("token_count", 0)) for c in chunks]
    print(f"Token per chunk: min={min(token_counts)}, max={max(token_counts)}, media={sum(token_counts)/len(token_counts):.1f}")

{"ts": "2026-03-12T10:50:11", "logger": "src.ingestion.pdf_loader", "level": "INFO", "message": "Chunked 9 documents into 9 chunks (chunk_size=512, overlap=64)"}


Parametri di chunking:
  chunk_size:    512 token
  chunk_overlap: 64 token

Chunks prodotti: 9 (da 9 documenti)
Token per chunk: min=12, max=76, media=53.8


In [8]:
# Ispezione dettagliata dei primi chunk
print("Dettaglio dei primi 5 chunk:")
print("=" * 70)
for chunk in chunks[:5]:
    text_preview = chunk.text[:120].replace("\n", " ")
    print(f"[chunk_index={chunk.chunk_index}]")
    print(f"  source={chunk.metadata.get('source')}, page={chunk.metadata.get('page')}, tokens={chunk.metadata.get('token_count')}")
    print(f"  text: '{text_preview}'")
    print()

Dettaglio dei primi 5 chunk:
[chunk_index=0]
  source=business_glossary.pdf, page=1, tokens=12
  text: 'BUSINESS GLOSSARY ? E-COMMERCE DOMAIN'

[chunk_index=1]
  source=business_glossary.pdf, page=2, tokens=56
  text: 'Customer A Customer is any individual or legal entity that has registered an account on the platform and has made at lea'

[chunk_index=2]
  source=business_glossary.pdf, page=3, tokens=63
  text: 'Product A Product is a tangible or digital item offered for sale on the platform. Each Product is uniquely identified by'

[chunk_index=3]
  source=business_glossary.pdf, page=4, tokens=76
  text: 'SalesOrder A SalesOrder is a formal transactional document that records the agreement between the platform and a Custome'

[chunk_index=4]
  source=business_glossary.pdf, page=5, tokens=62
  text: 'OrderLineItem An OrderLineItem is a single line within a SalesOrder that specifies a particular Product, the quantity or'



In [9]:
# Verifica su tutti i chunk
for chunk in chunks:
    assert chunk.chunk_index >= 0
    assert "source" in chunk.metadata
    assert "page" in chunk.metadata
    assert "token_count" in chunk.metadata
    assert int(chunk.metadata["token_count"]) <= settings.chunk_size, \
        f"chunk {chunk.chunk_index} supera chunk_size: {chunk.metadata['token_count']} > {settings.chunk_size}"

print(f"✅ Tutti i {len(chunks)} chunk:")
print(f"   · hanno chunk_index progressivo")
print(f"   · ereditano metadati source e page")
print(f"   · hanno token_count ≤ chunk_size ({settings.chunk_size})")

✅ Tutti i 9 chunk:
   · hanno chunk_index progressivo
   · ereditano metadati source e page
   · hanno token_count ≤ chunk_size (512)


## 4. `load_and_chunk_pdf()` — Convenience function

La funzione `load_and_chunk_pdf()` combina `load_pdf()` e `chunk_documents()` in un'unica chiamata, producendo direttamente i chunk pronti per l'SLM.

In [10]:
from src.ingestion.pdf_loader import load_and_chunk_pdf

# Singola chiamata: load + chunk
all_chunks = load_and_chunk_pdf(pdf_path)

# Deve produrre lo stesso risultato della pipeline separata
assert len(all_chunks) == len(chunks), "I chunk devono essere identici a quelli della pipeline separata"
print(f"load_and_chunk_pdf(): {len(all_chunks)} chunks ✅")
print(f"Identici a load_pdf() + chunk_documents(): {len(chunks)} chunks")

{"ts": "2026-03-12T10:50:11", "logger": "src.ingestion.pdf_loader", "level": "INFO", "message": "Loaded 9 pages from 'business_glossary.pdf'"}
{"ts": "2026-03-12T10:50:11", "logger": "src.ingestion.pdf_loader", "level": "INFO", "message": "Chunked 9 documents into 9 chunks (chunk_size=512, overlap=64)"}


load_and_chunk_pdf(): 9 chunks ✅
Identici a load_pdf() + chunk_documents(): 9 chunks


## 5. Visualizzazione: distribuzione dei token per chunk

In [11]:
# Distribuzione token senza dipendenze grafiche esterne
token_counts = [int(c.metadata.get("token_count", 0)) for c in chunks]

# Istogramma ASCII
buckets = {"0-64": 0, "65-128": 0, "129-256": 0, "257-384": 0, "385-512": 0}
for t in token_counts:
    if t <= 64: buckets["0-64"] += 1
    elif t <= 128: buckets["65-128"] += 1
    elif t <= 256: buckets["129-256"] += 1
    elif t <= 384: buckets["257-384"] += 1
    else: buckets["385-512"] += 1

max_count = max(buckets.values()) if buckets.values() else 1
bar_width = 40

print("Distribuzione token per chunk (istogramma ASCII):")
print()
for label, count in buckets.items():
    bar = "█" * int(count / max_count * bar_width)
    print(f"  {label:9s} | {bar:<40s} {count}")
print()
print(f"  Totale chunk: {len(token_counts)}")
print(f"  Media:        {sum(token_counts)/len(token_counts):.1f} token")
print(f"  Min:          {min(token_counts)} token")
print(f"  Max:          {max(token_counts)} token")
print(f"  chunk_size:   {settings.chunk_size} token (limite configurato)")

Distribuzione token per chunk (istogramma ASCII):

  0-64      | ████████████████████████████████████████ 7
  65-128    | ███████████                              2
  129-256   |                                          0
  257-384   |                                          0
  385-512   |                                          0

  Totale chunk: 9
  Media:        53.8 token
  Min:          12 token
  Max:          76 token
  chunk_size:   512 token (limite configurato)


In [12]:
# Cleanup
import shutil
shutil.rmtree(tmp_dir)
print(f"Directory temporanea rimossa: {tmp_dir}")

Directory temporanea rimossa: /tmp/tmp68wiqx6l


## Riepilogo — Architettura di Part 3 (Ingestion)

```
PDF file
   │
   ▼
load_pdf(path: Path) → list[Document]
   │  · fitz.open() → pagina per pagina
   │  · page.get_text("text") + strip()
   │  · metadata: {source, page}
   │  · IngestionError su: missing / encrypted / corrupt
   │
   ▼
chunk_documents(docs) → list[Chunk]
   │  · RecursiveCharacterTextSplitter
   │  · separators: ["\n\n", "\n", ". ", " "]
   │  · length_function: tiktoken cl100k_base
   │  · chunk_size=512, chunk_overlap=64 (da settings)
   │  · metadata: {source, page, token_count}
   │  · chunk_index progressivo globale
   │
   ▼
list[Chunk]  →  Extract_Triplets_SLM (Part 4)
```

**TASK-10 (DDL Parser) e TASK-11 (Schema Enricher)** non sono ancora implementati — i rispettivi file sono stub vuoti. Saranno dimostrati nel notebook di Part 3 completa quando implementati.